# Getters for mutable attributes

Imagine we're writing an application for a bakery (or really any kind of shop) that wants to keep track in which order their customers entered their shop, to ensure that customers get served in order.

To do this we create a `Queue` class. This class can keep track of a various number of `customers` (in this case simply represented by a list strings). And because we don't want any people cutting in front of the line, we make it very explicit that

- when **a customer enters**, they're **added to the back of the list**
- when we **serve a new customer**, we select the customer **that is currently at the front of the list** (and we remove them from the list as they are no longer in line).

Because we don't accidentally want to add any customers at various position in the list `customers`, we turn it into a private attribute.

Finally, we also want a getter method `get_customers()`, that shows us all the customers that are still currently waiting in line.

In [ ]:
class Queue:
    def __init__(self):
        self._customers = []

    def enter(self, name):
        self._customers.append(name)

    def next(self):
        return self._customers.pop(0)

    def get_customers(self):
        return self._customers

which we could use as follows:

In [ ]:
queue = Queue()
queue.enter("James")
queue.enter("Alice")
queue.enter("Bob")
print(f"Up next: {queue.next()}")
print(f"Still waiting: {queue.get_customers()}")

This class seems to work great, and because we made `_customers` a private attribute, we also know that we shouldn't do something like

In [ ]:
queue._customers = ["Bob", "Alice"] # switching the names around

or

In [ ]:
queue._customers.pop(1) # removing an element from the list that is not the first one

The fact that we are using an attribute starting with an underscore `_` should tell us immediately that we're doing something we shouldn't be doing!

However, take a look at and think about what the following code does:

In [ ]:
queue = Queue()
queue.enter("John")
queue.enter("Paul")
queue.enter("Simon")
print(f"Remaining in the queue: {queue.get_customers()}")
customers = queue.get_customers()
customers.pop(1)
print(f"Remaining in the queue: {queue.get_customers()}")

It seems that even without accessing the private attribute `_customers` we could still accidentally change the contents or order of the list of customers, while the entire point of our `Queue` class was that wouldn't be able to do that!

So first we have to ask ourselves: why is this happening?

Well the reason is that the `list`-object that `get_customers()` returns, is **the same object** as the one that is saved in the private attribute `_customers` and that **this object can change** (for example when `pop`ping an element). So when we pop an element from the list that we got from `get_customers()`, it is also popped from the list that is assigned to `_customers` (because it's the same list).

So how to fix this?

Well we only need to make sure that the list that `get_customers()` returns **is not the same list object** as the one assigned to `_customers`. We can achieve that by making a copy of the list in `_customers`. The easiest way to make a copy of a list is to pass that list as an argument to the list constructor:

In [ ]:
my_list = [1, 2, 3]
fake_copy = my_list
print(f"my_list: {my_list}")
print(f"fake_copy: {fake_copy}")
print(f"my_list the same object as fake_copy: {my_list is fake_copy}")
print(f"my_list has the same values as fake_copy: {my_list == fake_copy}")
print("------")
real_copy = list(my_list)
print(f"real copy: {real_copy}")
print(f"my_list the same object as real_copy: {my_list is real_copy}")
print(f"my_list has the same values as real_copy: {my_list == real_copy}")

As you may remember we can use the `is` operator to check whether two values are the exact same object, while `==` simply checks whether the two lists have the same values.

Knowing this, we can implement `get_customers()` to return a copy of the list in `_customers`:

In [ ]:
class Queue:
    def __init__(self):
        self._customers = []

    def enter(self, name):
        self._customers.append(name)

    def next(self):
        return self._customers.pop(0)

    def get_customers(self):
        return list(self._customers)

queue = Queue()
queue.enter("John")
queue.enter("Paul")
queue.enter("Simon")
customers = queue.get_customers()
customers.pop(1)
print(f"customers: {customers}")
print(f"customers in queue: {queue.get_customers()}")

Now it's no longer possible to change the private attribute `_customers` without accessing that attribute directly.

Note: `get_customers()` return **a new copy** every time it is called:

In [ ]:
queue.get_customers() is queue.get_customers() # these are not the same objects, a different list is created every time the method is called

## Attributes that contain mutable values

What we encountered in the previous sections applies any time the value of a private attribute is **a mutable value**, such as a list or any object that allows you to modify it.

Notably it doesn't apply to strings, numbers, booleans etc, as you cannot change these values themselves, you can only assign new values. If you need a refresher on mutable vs immutable objects, revisit the topic "mutable objects" in week 3.

## Exercises

### Exercise 84.1: Hands off my playlist

You're adding playlists to a music player. A playlist has a **name** and keeps its **songs** in a specific order — the order in which they were added. Other parts of the program may want to look at the songs (to display them, count them, ...), but they should never be able to mess with the playlist itself: no removing songs, no reordering.

Write a class `Playlist` with a public attribute `name` and a private attribute `_songs` that starts as an empty list, and two methods:

- `add_song(song)` adds a song to the end of the playlist
- `get_songs()` returns the songs in the playlist — but in such a way that whoever receives them cannot change the playlist itself

```python .noeval
playlist = Playlist("Road Trip")
playlist.add_song("Hey Jude")
playlist.add_song("Let It Be")

songs = playlist.get_songs()
print(songs) # ['Hey Jude', 'Let It Be']

songs.pop(0) # someone tampers with the list they got
print(playlist.get_songs()) # ['Hey Jude', 'Let It Be'] <- the playlist is unchanged!
```

Implement this class in the file [exercise_84_1_playlist.py](concept-exercises/exercise_84_1_playlist.py).
Run the following cell to verify that your implementation is correct:

In [ ]:
# code to run the tests
!python3 -m pytest -q --tb=short concept-exercises/.tests/test_exercise_84_1_playlist.py